In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KDTree
import time

In [ ]:
np.random.seed(60)
X = np.random.rand(10, 2)
print("Generated Points:")
print(X)
print("Shape of X:", X.shape)

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], s=80)
for i, (x, y) in enumerate(X):
    plt.text(x + 0.01, y + 0.01, f"P{i}")
plt.xlabel("X-coordinate")
plt.ylabel("Y-coordinate")
plt.title("Generated 2D Data Points")
plt.grid(True)
plt.show()

In [ ]:
D2 = np.sum((X[:, np.newaxis, :] - X[np.newaxis, :, :]) ** 2, axis=2)
print(D2)

In [ ]:
diagonal = np.diag(D2)
print("Diagonal values:", diagonal)
print("Are all diagonal values zero?", np.allclose(diagonal, 0))

In [ ]:
neighbor_order = np.argsort(D2, axis=1)
print("Neighbor ordering:")
print(neighbor_order)

In [ ]:
K = 2
nearest_neighbours = neighbor_order[:, 1:K+1]
print("Two nearest neighbours:")
print(nearest_neighbours)

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], s=80)
for i, (x, y) in enumerate(X):
    plt.text(x + 0.01, y + 0.01, f"P{i}")
    for j in nearest_neighbours[i]:
        plt.plot([X[i, 0], X[j, 0]], [X[i, 1], X[j, 1]])
plt.xlabel("X-coordinate")
plt.ylabel("Y-coordinate")
plt.title("Two-Nearest-Neighbor Network")
plt.grid(True)
plt.show()

In [ ]:
def loop_based_neighbours(X, K=2):
    N = len(X)
    distances = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            difference = X[i] - X[j]
            distances[i, j] = np.sum(difference ** 2)
    neighbours = np.zeros((N, K), dtype=int)
    for i in range(N):
        order = np.argsort(distances[i])
        neighbours[i] = order[1:K+1]
    return neighbours

In [ ]:
def vectorised_neighbours(X, K=2):
    D2 = np.sum((X[:, np.newaxis, :] - X[np.newaxis, :, :]) ** 2, axis=2)
    candidates = np.argpartition(D2, K + 1, axis=1)[:, :K + 1]
    candidate_distances = np.take_along_axis(D2, candidates, axis=1)
    order = np.argsort(candidate_distances, axis=1)
    sorted_candidates = np.take_along_axis(candidates, order, axis=1)
    return sorted_candidates[:, 1:K+1]

In [ ]:
for N in [100, 1000, 5000]:
    np.random.seed(60)
    X_test = np.random.rand(N, 2)
    start = time.perf_counter()
    result = vectorised_neighbours(X_test, K=2)
    elapsed = time.perf_counter() - start
    print(f"N = {N}, Vectorised time = {elapsed:.6f} seconds")

In [ ]:
loop_result = loop_based_neighbours(X, K=2)
vectorised_result = vectorised_neighbours(X, K=2)
print("Loop-based result:")
print(loop_result)
print("Vectorised result:")
print(vectorised_result)
print("Are both results identical?", np.array_equal(loop_result, vectorised_result))

In [ ]:
D2 = np.sum((X[:, np.newaxis, :] - X[np.newaxis, :, :]) ** 2, axis=2)
neighbor_order = np.argsort(D2, axis=1)
nearest_neighbours = neighbor_order[:, 1:3]
print("Nearest neighbors:")
print(nearest_neighbours)

In [ ]:
tree = KDTree(X)
distances, indices = tree.query(X, k=3)
print("KD-Tree distances:")
print(distances)
print("KD-Tree indices:")
print(indices)
print("Two nearest neighbors using KD-Tree:")
print(indices[:, 1:])